# Telco Customer Churn — Baseline (Logistic Regression)

**Goal.** Establish a clean, honest baseline for churn prediction.

**What this notebook does**
1. Loads and cleans the Telco churn dataset.
2. Performs **exploratory data analysis** focused on churn drivers (Contract, tenure, InternetService, PaymentMethod, monthly charges).
3. Splits the data **first**, then fits the scaler on the training fold only (no leakage).
4. Trains **logistic regression** with `class_weight='balanced'` to handle the ~26% positive-class imbalance.
5. Reports accuracy, precision, recall, **F1 (churn)**, ROC-AUC, PR-AUC and a confusion matrix.
6. Saves every figure to `figures/` and the final metrics row to `figures/baseline_metrics.csv` so they can be used directly on the symposium poster.

**Why this is the right baseline.** Logistic regression is interpretable, fast, and known to be very competitive on one-hot encoded tabular data. Beating it meaningfully is the bar for the MLP and boosted models.

In [ ]:
# Imports and seed
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

## 1. Load and clean

In [ ]:
df_raw = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Raw shape:", df_raw.shape)

df = df_raw.drop(columns=["customerID"]).copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
n_dropped = df["TotalCharges"].isna().sum()
df = df.dropna().reset_index(drop=True)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(f"Dropped {n_dropped} rows with invalid TotalCharges.")
print("Clean shape:", df.shape)
print(f"Churn rate: {df['Churn'].mean():.4f}")
df.head()

## 2. Exploratory Data Analysis

We focus on the columns most likely to drive churn so the poster has a clear visual story:
- Class balance (target)
- Contract type vs churn rate
- Tenure distribution split by churn
- Internet service type vs churn rate
- Payment method vs churn rate
- Monthly charges distribution split by churn
- Numeric correlation heatmap

Each figure is saved to `figures/`.

In [ ]:
# 2a. Class balance
fig, ax = plt.subplots(figsize=(5, 4))
counts = df["Churn"].value_counts().sort_index()
ax.bar(["Stayed (0)", "Churned (1)"], counts.values,
       color=["#4C72B0", "#DD8452"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, f"{v}\n({v/len(df):.1%})", ha="center")
ax.set_ylabel("Customers")
ax.set_title(f"Class balance — total {len(df):,} customers")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_01_class_balance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 2b. Churn rate by Contract
fig, ax = plt.subplots(figsize=(6, 4))
ct = df.groupby("Contract")["Churn"].mean().sort_values(ascending=False)
ct.plot(kind="bar", ax=ax, color="#DD8452")
ax.set_ylabel("Churn rate")
ax.set_title("Churn rate by contract type")
ax.axhline(df["Churn"].mean(), ls="--", color="gray", label=f"Overall {df['Churn'].mean():.2f}")
for i, v in enumerate(ct.values):
    ax.text(i, v + 0.01, f"{v:.2f}", ha="center")
ax.legend(); plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_02_churn_by_contract.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 2c. Tenure distribution split by churn
fig, ax = plt.subplots(figsize=(7, 4))
for label, color in [(0, "#4C72B0"), (1, "#DD8452")]:
    sub = df.loc[df["Churn"] == label, "tenure"]
    ax.hist(sub, bins=30, alpha=0.6, label=f"Churn={label}", color=color)
ax.set_xlabel("tenure (months)")
ax.set_ylabel("Customers")
ax.set_title("Tenure distribution — churners are concentrated in the first ~12 months")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_03_tenure_hist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 2d. Churn rate by InternetService
fig, ax = plt.subplots(figsize=(6, 4))
ct = df.groupby("InternetService")["Churn"].mean().sort_values(ascending=False)
ct.plot(kind="bar", ax=ax, color="#55A868")
ax.set_ylabel("Churn rate")
ax.set_title("Churn rate by internet service")
ax.axhline(df["Churn"].mean(), ls="--", color="gray", label=f"Overall {df['Churn'].mean():.2f}")
for i, v in enumerate(ct.values):
    ax.text(i, v + 0.01, f"{v:.2f}", ha="center")
ax.legend(); plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_04_churn_by_internet.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 2e. Churn rate by PaymentMethod
fig, ax = plt.subplots(figsize=(7, 4))
ct = df.groupby("PaymentMethod")["Churn"].mean().sort_values(ascending=False)
ct.plot(kind="bar", ax=ax, color="#C44E52")
ax.set_ylabel("Churn rate")
ax.set_title("Churn rate by payment method")
ax.axhline(df["Churn"].mean(), ls="--", color="gray", label=f"Overall {df['Churn'].mean():.2f}")
for i, v in enumerate(ct.values):
    ax.text(i, v + 0.01, f"{v:.2f}", ha="center")
ax.legend(); plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_05_churn_by_payment.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 2f. MonthlyCharges distribution split by churn
fig, ax = plt.subplots(figsize=(7, 4))
for label, color in [(0, "#4C72B0"), (1, "#DD8452")]:
    sub = df.loc[df["Churn"] == label, "MonthlyCharges"]
    ax.hist(sub, bins=30, alpha=0.6, label=f"Churn={label}", color=color)
ax.set_xlabel("MonthlyCharges ($)")
ax.set_ylabel("Customers")
ax.set_title("MonthlyCharges — churners skew toward higher bills")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_06_monthly_charges_hist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 2g. Correlation heatmap (numeric + encoded target)
df_num = df.copy()
df_num = pd.get_dummies(df_num, drop_first=True).astype(float)
corr = df_num.corr()["Churn"].drop("Churn").sort_values()
fig, ax = plt.subplots(figsize=(6, 8))
corr.plot(kind="barh", ax=ax,
          color=["#4C72B0" if v < 0 else "#DD8452" for v in corr.values])
ax.set_title("Correlation of each feature with Churn (after one-hot)")
ax.set_xlabel("Pearson correlation with Churn")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_07_corr_with_churn.png", dpi=150, bbox_inches="tight")
plt.show()

**EDA takeaways (for the poster narrative)**
- Class is **imbalanced** (~26% churn) → use F1 / AUC, not accuracy.
- **Month-to-month** contracts churn dramatically more than 1- or 2-year contracts.
- **Short-tenure** customers (first ~12 months) dominate the churn population.
- **Fiber optic** internet has a much higher churn rate than DSL or no internet.
- **Electronic check** payers churn far more than auto-pay or mailed-check customers.
- Higher **MonthlyCharges** correlate with higher churn.

## 3. Encode, split, scale (leakage-free)

**Order matters.** We split **before** scaling and fit `StandardScaler` only on the training fold. The previous version of this notebook fit the scaler on the full feature matrix; that is leakage and is corrected here.

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
X = df_encoded.drop(columns=["Churn"])
y = df_encoded["Churn"].astype(int)
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Train:", X_train_s.shape, "  Test:", X_test_s.shape)
print(f"Train churn rate: {y_train.mean():.4f}   Test churn rate: {y_test.mean():.4f}")

## 4. Logistic regression baseline

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
model.fit(X_train_s, y_train)

y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

metrics = {
    "model": "Logistic Regression (baseline)",
    "accuracy":  accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall":    recall_score(y_test, y_pred),
    "f1":        f1_score(y_test, y_pred),
    "roc_auc":   roc_auc_score(y_test, y_proba),
    "pr_auc":    average_precision_score(y_test, y_proba),
}
for k, v in metrics.items():
    print(f"{k:>10}: {v if isinstance(v, str) else f'{v:.4f}'}")
print()
print(classification_report(y_test, y_pred, digits=4))

pd.DataFrame([metrics]).to_csv(f"{FIG_DIR}/baseline_metrics.csv", index=False)
print(f"Saved -> {FIG_DIR}/baseline_metrics.csv")

## 5. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Stayed", "Churned"], yticklabels=["Stayed", "Churned"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Logistic Regression — confusion matrix (test)\nF1={metrics['f1']:.3f}  ROC-AUC={metrics['roc_auc']:.3f}")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/baseline_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Most influential coefficients (interpretable baseline)

In [ ]:
coefs = pd.Series(model.coef_[0], index=feature_names).sort_values()
top_neg = coefs.head(10)   # push toward staying
top_pos = coefs.tail(10)   # push toward churn

fig, ax = plt.subplots(figsize=(7, 7))
combo = pd.concat([top_neg, top_pos])
combo.plot(kind="barh", ax=ax,
           color=["#4C72B0" if v < 0 else "#DD8452" for v in combo.values])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Logistic regression coefficient")
ax.set_title("Top features pushing toward Churn (orange) vs Stay (blue)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/baseline_top_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. What this baseline tells us
- Logistic regression with `class_weight='balanced'` already reaches a credible ROC-AUC and churn-F1 (see `baseline_metrics.csv`).
- The **direction** of the top coefficients matches the EDA: month-to-month contract, fiber optic, electronic check, short tenure, and high MonthlyCharges all push toward churn.
- The MLP and boosted models in the other notebooks must beat this *meaningfully* (not just by a fraction) to be worth presenting as upgrades.